## Imports and dataset discovery

In [1]:
# ============================================
# Cell 1 — Imports and dataset discovery
# ============================================
import os, time, math, random, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'CUDA: {torch.cuda.get_device_name(0)}')

INPUT_ROOT = Path('/kaggle/input')
print(f'\nAttached datasets:')
for d in INPUT_ROOT.iterdir():
    print(f'  {d.name}')

h5_files = list(INPUT_ROOT.rglob('*.h5'))
print(f'\nFound {len(h5_files)} .h5 files:')
for f in h5_files:
    size_mb = f.stat().st_size / 1024**2
    print(f'  {size_mb:7.1f} MB  {f}')

Device: cuda
CUDA: Tesla T4

Attached datasets:
  datasets

Found 2 .h5 files:
    241.7 MB  /kaggle/input/datasets/vathsalupadhyay/scanobjectnn-pb-t50-rs/training_objectdataset_augmentedrot_scale75.h5
     60.1 MB  /kaggle/input/datasets/vathsalupadhyay/scanobjectnn-pb-t50-rs/test_objectdataset_augmentedrot_scale75.h5


## Loading training and test HDF5

In [2]:
# ============================================
# Cell 2 — Load training and test HDF5 files
# ============================================
def find_h5(patterns):
    for p in patterns:
        for f in h5_files:
            if p in f.name.lower():
                return f
    return None

TRAIN_PATTERNS = [
    'training_objectdataset_augmentedrot_scale75',
    'training_objectdataset',
    'ply_data_train',
    'train',
]
TEST_PATTERNS = [
    'test_objectdataset_augmentedrot_scale75',
    'test_objectdataset',
    'ply_data_test',
    'test',
]

train_path = find_h5(TRAIN_PATTERNS)
test_path  = find_h5(TEST_PATTERNS)
assert train_path is not None, 'No training .h5 found.'
assert test_path  is not None, 'No test .h5 found.'
print(f'TRAIN: {train_path}')
print(f'TEST:  {test_path}')

def load_h5(p):
    with h5py.File(p, 'r') as f:
        data   = f['data'][:].astype(np.float32)
        labels = f['label'][:].astype(np.int64).flatten()
    return data, labels

train_data, train_labels = load_h5(train_path)
test_data,  test_labels  = load_h5(test_path)

print(f'\nTrain: data {train_data.shape}, labels {train_labels.shape}, classes {train_labels.min()}-{train_labels.max()}')
print(f'Test:  data {test_data.shape},  labels {test_labels.shape},  classes {test_labels.min()}-{test_labels.max()}')

SCANOBJ_CLASSES = [
    'bag', 'bin', 'box', 'cabinet', 'chair', 'desk', 'display', 'door',
    'shelf', 'table', 'bed', 'pillow', 'sink', 'sofa', 'toilet'
]
NUM_CLASSES = len(SCANOBJ_CLASSES)

unique, counts = np.unique(train_labels, return_counts=True)
print('\nClass distribution (train):')
for u, c in zip(unique, counts):
    print(f'  {u:2d} {SCANOBJ_CLASSES[u]:10s}: {c:5d}')

TRAIN: /kaggle/input/datasets/vathsalupadhyay/scanobjectnn-pb-t50-rs/training_objectdataset_augmentedrot_scale75.h5
TEST:  /kaggle/input/datasets/vathsalupadhyay/scanobjectnn-pb-t50-rs/test_objectdataset_augmentedrot_scale75.h5

Train: data (11416, 2048, 3), labels (11416,), classes 0-14
Test:  data (2882, 2048, 3),  labels (2882,),  classes 0-14

Class distribution (train):
   0 bag       :   298
   1 bin       :   794
   2 box       :   406
   3 cabinet   :  1344
   4 chair     :  1585
   5 desk      :   592
   6 display   :   678
   7 door      :   892
   8 shelf     :  1084
   9 table     :   922
  10 bed       :   564
  11 pillow    :   405
  12 sink      :   469
  13 sofa      :  1058
  14 toilet    :   325


## PointNet++ MSG architecture

In [3]:
# ============================================
# Cell 3 — PointNet++ MSG architecture
# ============================================
def farthest_point_sample(xyz, npoint):
    B, N, _ = xyz.shape
    dev = xyz.device
    centroids = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    distance = torch.full((B, N), float("inf"), device=dev)
    farthest = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
    batch_idx = torch.arange(B, dtype=torch.long, device=dev)
    for i in range(npoint):
        centroids[:, i] = farthest
        cxyz = xyz[batch_idx, farthest, :].unsqueeze(1)
        dist = ((xyz - cxyz) ** 2).sum(dim=-1)
        distance = torch.minimum(distance, dist)
        farthest = distance.argmax(dim=-1)
    return centroids


def index_points(points, idx):
    B = points.shape[0]
    vs = list(idx.shape); vs[1:] = [1]*(len(vs)-1)
    rs = list(idx.shape); rs[0] = 1
    bi = torch.arange(B, dtype=torch.long, device=points.device).view(vs).repeat(rs)
    return points[bi, idx, :]


def ball_query(radius, nsample, xyz, new_xyz):
    B, N, _ = xyz.shape
    _, S, _ = new_xyz.shape
    dev = xyz.device
    gi = torch.arange(N, dtype=torch.long, device=dev).view(1, 1, N).repeat(B, S, 1)
    sd = ((new_xyz.unsqueeze(2) - xyz.unsqueeze(1)) ** 2).sum(dim=-1)
    gi[sd > radius ** 2] = N
    gi = gi.sort(dim=-1)[0][:, :, :nsample]
    gf = gi[:, :, 0:1].repeat(1, 1, nsample); gi[gi == N] = gf[gi == N]
    return gi


class SetAbstractionMSG(nn.Module):
    def __init__(self, npoint, radii, nsamples, in_channel, mlps):
        super().__init__()
        self.npoint, self.radii, self.nsamples = npoint, radii, nsamples
        self.conv_blocks, self.bn_blocks = nn.ModuleList(), nn.ModuleList()
        for mlp in mlps:
            convs, bns = nn.ModuleList(), nn.ModuleList()
            last = in_channel + 3
            for c in mlp:
                convs.append(nn.Conv2d(last, c, 1)); bns.append(nn.BatchNorm2d(c)); last = c
            self.conv_blocks.append(convs); self.bn_blocks.append(bns)

    def forward(self, xyz, features=None):
        fps = farthest_point_sample(xyz, self.npoint)
        new_xyz = index_points(xyz, fps); outs = []
        for i, (r, k) in enumerate(zip(self.radii, self.nsamples)):
            nn_idx = ball_query(r, k, xyz, new_xyz)
            g_xyz = index_points(xyz, nn_idx) - new_xyz.unsqueeze(2)
            if features is not None:
                g = torch.cat([g_xyz, index_points(features, nn_idx)], dim=-1)
            else:
                g = g_xyz
            g = g.permute(0, 3, 1, 2).contiguous()
            for conv, bn in zip(self.conv_blocks[i], self.bn_blocks[i]):
                g = F.relu(bn(conv(g)))
            outs.append(g.max(dim=-1)[0])
        return new_xyz, torch.cat(outs, dim=1).permute(0, 2, 1).contiguous()


class GlobalSetAbstraction(nn.Module):
    def __init__(self, in_channel, mlp):
        super().__init__()
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        last = in_channel
        for c in mlp:
            self.convs.append(nn.Conv1d(last, c, 1)); self.bns.append(nn.BatchNorm1d(c)); last = c

    def forward(self, xyz, features):
        x = torch.cat([xyz, features], dim=-1).permute(0, 2, 1)
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x)))
        return x.max(dim=-1)[0]


class PointNetPlusPlusMSG(nn.Module):
    def __init__(self, num_classes=15, dropout=0.5):
        super().__init__()
        self.sa1 = SetAbstractionMSG(512, [0.1, 0.2, 0.4], [16, 32, 128], 0,
                                     [[32, 32, 64], [64, 64, 128], [64, 96, 128]])
        self.sa2 = SetAbstractionMSG(128, [0.2, 0.4, 0.8], [32, 64, 128], 320,
                                     [[64, 64, 128], [128, 128, 256], [128, 128, 256]])
        self.sa_global = GlobalSetAbstraction(640 + 3, [256, 512, 1024])
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,  num_classes),
        )

    def forward(self, xyz):
        l1_xyz, l1_f = self.sa1(xyz, None)
        l2_xyz, l2_f = self.sa2(l1_xyz, l1_f)
        return self.classifier(self.sa_global(l2_xyz, l2_f))


print('PointNet++ MSG defined for 15-class ScanObjectNN.')

PointNet++ MSG defined for 15-class ScanObjectNN.


## Augmentation utilities

In [4]:
# ============================================
# Cell 4 — Augmentation utilities
# ============================================
def random_rotation_z(points):
    theta = np.random.uniform(0, 2 * np.pi)
    cos, sin = np.cos(theta), np.sin(theta)
    R = np.array([[cos, -sin, 0],
                  [sin,  cos, 0],
                  [0,    0,   1]], dtype=np.float32)
    return points @ R.T

def random_scale(points, lo=0.8, hi=1.25):
    return points * np.random.uniform(lo, hi)

def random_translate(points, t=0.1):
    return points + np.random.uniform(-t, t, size=(1, 3)).astype(np.float32)

def jitter(points, sigma=0.01, clip=0.05):
    noise = np.clip(sigma * np.random.randn(*points.shape), -clip, clip).astype(np.float32)
    return points + noise

def random_point_dropout(points, max_dropout=0.125):
    dropout_ratio = np.random.uniform(0, max_dropout)
    drop_idx = np.where(np.random.random(len(points)) <= dropout_ratio)[0]
    if len(drop_idx) > 0:
        points = points.copy()
        points[drop_idx] = points[0]
    return points

def normalize_unit_sphere(points):
    points = points - points.mean(axis=0, keepdims=True)
    max_dist = np.linalg.norm(points, axis=1).max()
    points = points / (max_dist + 1e-8)
    return points

print('Augmentation utilities defined.')

Augmentation utilities defined.


## Dataset (random subsample, num_workers=0)

In [5]:
# ============================================
# Cell 5 — Dataset class (fast random subsample)
# ============================================
class ScanObjectNNDataset(Dataset):
    def __init__(self, data, labels, n_points=1024, training=True):
        self.data = data
        self.labels = labels
        self.n_points = n_points
        self.training = training

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        points = self.data[idx].copy()

        if points.shape[0] != self.n_points:
            choice = np.random.choice(points.shape[0], self.n_points, replace=False)
            points = points[choice]

        points = normalize_unit_sphere(points)

        if self.training:
            points = random_rotation_z(points)
            points = random_scale(points)
            points = random_translate(points)
            points = jitter(points)
            points = random_point_dropout(points)
            points = normalize_unit_sphere(points)

        return torch.from_numpy(points.astype(np.float32)), int(self.labels[idx])


N_POINTS   = 1024
BATCH_SIZE = 32

train_ds = ScanObjectNNDataset(train_data, train_labels, n_points=N_POINTS, training=True)
test_ds  = ScanObjectNNDataset(test_data,  test_labels,  n_points=N_POINTS, training=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, drop_last=True,  pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, drop_last=False, pin_memory=True)

print(f'Train batches: {len(train_loader)}  ({len(train_ds)} samples)')
print(f'Test batches:  {len(test_loader)}   ({len(test_ds)} samples)')##

Train batches: 356  (11416 samples)
Test batches:  91   (2882 samples)


##  EMA and training utilities

In [6]:
# ============================================
# Cell 6 — EMA + train/eval helpers
# ============================================
class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
            else:
                self.shadow[k].copy_(v.detach())

    def apply(self, model):
        model.load_state_dict(self.shadow, strict=True)


def train_one_epoch(model, loader, optimizer, scaler, loss_fn, ema, device):
    model.train()
    total_loss, total, correct = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(xb)
            loss   = loss_fn(logits, yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        total_loss += loss.item() * xb.size(0)
        preds       = logits.argmax(dim=-1)
        correct    += (preds == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    per_class = defaultdict(lambda: [0, 0])
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        logits = model(xb)
        preds  = logits.argmax(dim=-1)
        correct += (preds == yb).sum().item()
        total   += xb.size(0)
        for t, p in zip(yb.tolist(), preds.tolist()):
            per_class[t][1] += 1
            if t == p:
                per_class[t][0] += 1
    return correct / total, dict(per_class)


print('Training utilities defined.')

Training utilities defined.


## Training setup with 60 Epochs

In [7]:
# ============================================
# Cell 7 — Training setup
# ============================================
EPOCHS = 60
LR_INIT = 1e-3
LR_MIN  = 1e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
EMA_DECAY = 0.999

model = PointNetPlusPlusMSG(num_classes=NUM_CLASSES, dropout=0.5).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {n_params/1e6:.2f}M')

optimizer = AdamW(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
loss_fn   = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler    = torch.amp.GradScaler('cuda')
ema       = ModelEMA(model, decay=EMA_DECAY)

est_hours = EPOCHS * 10 / 60
print(f'Setup OK. {EPOCHS} epochs × ~10 min = ~{est_hours:.1f} hours estimated.')
print(f'Batch {BATCH_SIZE}, init LR {LR_INIT}, EMA {EMA_DECAY}, AMP on.')

Model params: 1.74M
Setup OK. 60 epochs × ~10 min = ~10.0 hours estimated.
Batch 32, init LR 0.001, EMA 0.999, AMP on.


## Full training loop with checkpointing

In [8]:
# ============================================
# Cell 8 — Full training loop (60 epochs)
# ============================================
OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

best_test_acc = 0.0
history = []

for ep in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scaler, loss_fn, ema, device)

    eval_model = PointNetPlusPlusMSG(num_classes=NUM_CLASSES).to(device)
    ema.apply(eval_model)
    te_acc, te_pc = evaluate(eval_model, test_loader, device)

    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    elapsed = time.time() - t0
    print(f'Epoch {ep+1:3d}/{EPOCHS}  lr={current_lr:.6f}  train_loss={tr_loss:.4f}  '
          f'train_acc={tr_acc:.4f}  test_acc(EMA)={te_acc:.4f}  ({elapsed:.0f}s)',
          flush=True)

    history.append({
        'epoch':     ep + 1,
        'lr':        current_lr,
        'train_loss':tr_loss,
        'train_acc': tr_acc,
        'test_acc':  te_acc,
    })

    if te_acc > best_test_acc:
        best_test_acc = te_acc
        ckpt = {
            'epoch':   ep + 1,
            'model':   eval_model.state_dict(),
            'val_acc': te_acc,
            'is_ema':  True,
            'num_classes': NUM_CLASSES,
            'class_names': SCANOBJ_CLASSES,
        }
        torch.save(ckpt, OUT_DIR / 'pointnetpp_msg_scanobjectnn_best.pt')
        print(f'   --> New best, saved (test_acc={te_acc:.4f})', flush=True)

    if (ep + 1) % 10 == 0 or ep == EPOCHS - 1:
        torch.save({
            'epoch':   ep + 1,
            'model':   model.state_dict(),
            'val_acc': te_acc,
            'is_ema':  False,
        }, OUT_DIR / 'pointnetpp_msg_scanobjectnn_last.pt')

    if (ep + 1) % 5 == 0 or ep == EPOCHS - 1:
        with open(OUT_DIR / 'training_history.json', 'w') as f:
            json.dump(history, f, indent=2)

print(f'\nDone. Best test accuracy: {best_test_acc:.4f}', flush=True)

Epoch   1/60  lr=0.001000  train_loss=2.2638  train_acc=0.3043  test_acc(EMA)=0.0836  (255s)
   --> New best, saved (test_acc=0.0836)
Epoch   2/60  lr=0.000999  train_loss=2.0160  train_acc=0.4036  test_acc(EMA)=0.0836  (258s)
Epoch   3/60  lr=0.000997  train_loss=1.9080  train_acc=0.4500  test_acc(EMA)=0.0729  (258s)
Epoch   4/60  lr=0.000994  train_loss=1.8225  train_acc=0.4831  test_acc(EMA)=0.0729  (259s)
Epoch   5/60  lr=0.000989  train_loss=1.7609  train_acc=0.5131  test_acc(EMA)=0.0729  (258s)
Epoch   6/60  lr=0.000983  train_loss=1.7020  train_acc=0.5363  test_acc(EMA)=0.0729  (258s)
Epoch   7/60  lr=0.000976  train_loss=1.6525  train_acc=0.5654  test_acc(EMA)=0.0729  (259s)
Epoch   8/60  lr=0.000967  train_loss=1.6023  train_acc=0.5808  test_acc(EMA)=0.0763  (258s)
Epoch   9/60  lr=0.000957  train_loss=1.5589  train_acc=0.6018  test_acc(EMA)=0.1096  (258s)
   --> New best, saved (test_acc=0.1096)
Epoch  10/60  lr=0.000946  train_loss=1.5263  train_acc=0.6147  test_acc(EMA)=0.1

## Evaluation 

In [9]:
# ============================================
# Cell 9 — Final per-class evaluation
# ============================================
best_ckpt = torch.load(OUT_DIR / 'pointnetpp_msg_scanobjectnn_best.pt',
                       map_location=device, weights_only=False)
eval_model = PointNetPlusPlusMSG(num_classes=NUM_CLASSES).to(device)
eval_model.load_state_dict(best_ckpt['model'])
eval_model.eval()

test_acc, per_class = evaluate(eval_model, test_loader, device)
print(f'Best EMA test accuracy: {test_acc:.4f}\n')

print(f'{"class":12s}  {"correct":>8s} / {"total":>5s}   {"acc":>6s}')
print('-' * 45)
for i in range(NUM_CLASSES):
    if i in per_class:
        c, t = per_class[i]
        acc = c / t if t else 0
        print(f'{SCANOBJ_CLASSES[i]:12s}  {c:>8d} / {t:>5d}   {acc:>6.1%}')

mean_class_acc = np.mean([c/t for c, t in per_class.values() if t > 0])
print(f'\nOverall accuracy:        {test_acc:.4f}')
print(f'Mean per-class accuracy: {mean_class_acc:.4f}')

Best EMA test accuracy: 0.6766

class          correct / total      acc
---------------------------------------------
bag                 33 /    83    39.8%
bin                150 /   199    75.4%
box                 14 /   133    10.5%
cabinet            251 /   372    67.5%
chair              316 /   390    81.0%
desk                78 /   150    52.0%
display            140 /   204    68.6%
door               194 /   210    92.4%
shelf              183 /   241    75.9%
table              171 /   270    63.3%
bed                 70 /   110    63.6%
pillow              74 /   105    70.5%
sink                52 /   120    43.3%
sofa               169 /   210    80.5%
toilet              55 /    85    64.7%

Overall accuracy:        0.6766
Mean per-class accuracy: 0.6327
